<a href="https://colab.research.google.com/github/katerina-tech/WiDS-Global-Datathon-2026-Erope-Team/blob/eve/WiDS_2026_%7C_Get_Started_Europe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

widsworldwide_globaldathon26_path = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
hmnshudhmn24_wids_global_datathon_2026_path = kagglehub.notebook_output_download('hmnshudhmn24/wids-global-datathon-2026')
belbino_wildfire_scikitlearn_default_1_path = kagglehub.model_download('belbino/wildfire/ScikitLearn/default/1')

print('Data source import complete.')


 # **WiDS Global Datathon 2026**

# IMPORT LIBRARY

In [ ]:
import cloudpickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

```python
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
```

# DATA PREPROCESSING

## LOAD DATA

In [ ]:
meta_data_df = pd.read_csv(f"{widsworldwide_globaldathon26_path}/metaData.csv")
train_df = pd.read_csv(f"{widsworldwide_globaldathon26_path}/train.csv")
test_df = pd.read_csv(f"{widsworldwide_globaldathon26_path}/test.csv")

## EDA

In [ ]:

# 1. How many fires actually hit vs censored?
plt.figure(figsize=(6, 4))
sns.countplot(x='event', data=train_df, hue='event', palette='viridis', legend=False)
plt.title('Target Distribution: Hits (1) vs. Censored (0)')
plt.show()
# 2. When do the hits happen?
plt.figure(figsize=(8, 5))
sns.histplot(train_df[train_df['event'] == 1]['time_to_hit_hours'], bins=15, color='orange', kde=True)
plt.axvline(12, color='red', linestyle='--', label='12h Mark')
plt.axvline(24, color='red', linestyle='--', label='24h Mark')

plt.axvline(48, color='orange', linestyle='--', label='48h Mark')
plt.axvline(72, color='orange', linestyle='--', label='72h Mark')

plt.title('Distribution of Time-to-Hit (Only for Event=1)')
plt.legend()
plt.show()

MOST FIRES DO NOT hit, this could cause model to be imbalance.

For those that hits,most hit evacuation zones very early (within ~5 to 10 hours), while a smaller but still meaningful number occur within 24 hours. Hits after 24 hours are relatively rare.

The following step mainly to list down 5 important features and by grouping them based on hit vs censored, we are trying to see if there are any pattern, to see which variable might be explaining the difference in outcomes.

hit vs non-hit pattern

In [ ]:

killer_features = [
    'closing_speed_m_per_h',
    'dist_min_ci_0_5h',
    'area_growth_rate_ha_per_h',
    'alignment_abs',
    'num_perimeters_0_5h'
]

comparison_df = train_df.groupby('event')[killer_features].agg(['mean', 'median']).T
train_df['speed_dist_ratio'] = train_df['closing_speed_m_per_h'] / (train_df['dist_min_ci_0_5h'] + 1)

hits_only = train_df[train_df['event'] == 1]
time_correlations = hits_only[killer_features + ['time_to_hit_hours']].corr()['time_to_hit_hours'].sort_values()

display(train_df['event'].value_counts())

print("\n--- Summary of Time-to-Hit (Hits Only) ---\n")
display(train_df[train_df['event'] == 1]['time_to_hit_hours'].describe())

print("\n--- Numerical Comparison: No-Hit (0) vs Hit (1) ---\n")
display(comparison_df)

print("\n--- Speed/Distance Ratio Statistics ---\n")
display(train_df.groupby('event')['speed_dist_ratio'].describe())

print("\n--- Correlations with Time-to-Hit (Numerical) ---\n")
print(time_correlations)


## DATA PREPARATION

### TARGET COLUMNS

In [ ]:

# The competition horizons
horizons = [12, 24, 48, 72]

for h in horizons:
    train_df[f'target_{h}h'] = ((train_df['event'] == 1) & (train_df['time_to_hit_hours'] <= h)).astype(int)

# Check the imbalance for each
print("--- Positive Cases (Hits) per Horizon ---")
for h in horizons:
    count = train_df[f'target_{h}h'].sum()
    percentage = (count / len(train_df)) * 100
    print(f"{h}h Horizon: {count} hits ({percentage:.1f}%)")


--- Positive Cases (Hits) per Horizon ---
- 12h Horizon: 49 hits (22.2%)
- 24h Horizon: 63 hits (28.5%)
- 48h Horizon: 66 hits (29.9%)
- 72h Horizon: 69 hits (31.2%)

#### FEATURES & TARGET

In [ ]:
drop_cols = ['time_to_hit_hours', 'speed_dist_ratio',
    'event', 'target_12h', 'target_24h',
    'target_48h', 'target_72h', 'fold']

base_features = [col for col in train_df.columns if col not in drop_cols]

X=train_df[base_features]
y=train_df[['target_12h', 'target_24h','target_48h', 'target_72h']]


## FEATURE ENGINEERING

XGBoost builds many small decision trees. Each tree tries different feature conditions (e.g distance or speed), and only keeps the ones that help separate hits vs non-hits better.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if 'event_id' in X.columns:
            X = X.drop(columns=['event_id'])


        X['safe_closing_speed'] = X['closing_speed_m_per_h'].clip(lower=0)

        # Effective Speed
        X['effective_speed'] = (X['closing_speed_m_per_h'] * X['alignment_abs']).clip(lower=0)

        # Arrival Estimates
        X['simple_eta'] = X['dist_min_ci_0_5h'] / (X['safe_closing_speed'] + 1e-5)
        X['log_eta'] = np.log1p(X['dist_min_ci_0_5h'] / (X['effective_speed'] + 1))

        # Threat and Ratios
        X['speed_dist_ratio'] = X['safe_closing_speed'] / (X['dist_min_ci_0_5h'] + 1)
        X['threat_score'] = (X['effective_speed']) / (np.log1p(X['dist_min_ci_0_5h']) + 1)

        # Growth and Momentum
        X['growth_intensity'] = X['area_growth_abs_0_5h'] / (X['area_first_ha'] + 1)
        X['growth_momentum'] = X['area_growth_rate_ha_per_h'] * X['alignment_cos']

        # Temporal & Quality Features
        X['mapping_frequency'] = X['num_perimeters_0_5h'] / (X['dt_first_last_0_5h'] + 0.1)
        X['hour_sin'] = np.sin(2 * np.pi * X['event_start_hour'] / 24)
        X['hour_cos'] = np.cos(2 * np.pi * X['event_start_hour'] / 24)

        return X

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if 'event_id' in X.columns:
            X = X.drop(columns=['event_id'])


        X['safe_closing_speed'] = X['closing_speed_m_per_h'].clip(lower=0)

        # Effective Speed
        X['effective_speed'] = (X['closing_speed_m_per_h'] * X['alignment_abs']).clip(lower=0)

        # Arrival Estimates
        X['simple_eta'] = X['dist_min_ci_0_5h'] / (X['safe_closing_speed'] + 1e-5)
        X['log_eta'] = np.log1p(X['dist_min_ci_0_5h'] / (X['effective_speed'] + 1))

        # Threat and Ratios
        X['speed_dist_ratio'] = X['safe_closing_speed'] / (X['dist_min_ci_0_5h'] + 1)
        X['threat_score'] = (X['effective_speed']) / (np.log1p(X['dist_min_ci_0_5h']) + 1)

        # Growth and Momentum
        X['growth_intensity'] = X['area_growth_abs_0_5h'] / (X['area_first_ha'] + 1)
        X['growth_momentum'] = X['area_growth_rate_ha_per_h'] * X['alignment_cos']

        # Temporal & Quality Features
        X['mapping_frequency'] = X['num_perimeters_0_5h'] / (X['dt_first_last_0_5h'] + 0.1)
        X['hour_sin'] = np.sin(2 * np.pi * X['event_start_hour'] / 24)
        X['hour_cos'] = np.cos(2 * np.pi * X['event_start_hour'] / 24)

        return X


Safe closing speed: This feature keeps only the speed that is nearing the evacuation zone and turns everything that is negative to 0, ignoring fire movement away from it. Without clipping, model wastes effort learning noise, with clipping, model focuses on real signal.

Effective Speed: This feature adjusts the fire's speed based on how aligned it is toward the evacuation zone, capturing how effectively it is moving toward risk.

Log eta: This is a log-transformed version of estimated arrival time to reduce extreme values and make patterns easier for the model to learn.

Threat score: This feature combines effective speed and distance in log into a threat score, where distance is smoothed using a log transformation to reduce extreme differences. This is creating multiple ways to represent same concept (risk).

Growth & Momentum: small fire growing fast → dangerous big fire growing slowly → less aggressive This feature measures how aggresively the fire is growing relative to its initial size because absolute growth alone can be misleading, e.g +50 hectares means very different for small fire vs big fire

fast growth + toward evac → dangerous fast growth but sideways/away → less dangerous

Mapping frequency: This feature measures how frequently the fire perimeter is updated, which may reflect activity level or data quality.

Temporal: These features encode the hour of the day in a circular way so that similar times (like 23:00 and 00:00) are treated as close.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if 'event_id' in X.columns:
            X = X.drop(columns=['event_id'])


        X['safe_closing_speed'] = X['closing_speed_m_per_h'].clip(lower=0)

        # Effective Speed
        X['effective_speed'] = (X['closing_speed_m_per_h'] * X['alignment_abs']).clip(lower=0)

        # Arrival Estimates
        X['simple_eta'] = X['dist_min_ci_0_5h'] / (X['safe_closing_speed'] + 1e-5)
        X['log_eta'] = np.log1p(X['dist_min_ci_0_5h'] / (X['effective_speed'] + 1))

        # Threat and Ratios
        X['speed_dist_ratio'] = X['safe_closing_speed'] / (X['dist_min_ci_0_5h'] + 1)
        X['threat_score'] = (X['effective_speed']) / (np.log1p(X['dist_min_ci_0_5h']) + 1)

        # Growth and Momentum
        X['growth_intensity'] = X['area_growth_abs_0_5h'] / (X['area_first_ha'] + 1)
        X['growth_momentum'] = X['area_growth_rate_ha_per_h'] * X['alignment_cos']

        # Temporal & Quality Features
        X['mapping_frequency'] = X['num_perimeters_0_5h'] / (X['dt_first_last_0_5h'] + 0.1)
        X['hour_sin'] = np.sin(2 * np.pi * X['event_start_hour'] / 24)
        X['hour_cos'] = np.cos(2 * np.pi * X['event_start_hour'] / 24)

        return X

FeatureEngineer()

In [ ]:
#change1
fe = FeatureEngineer()
X_fe = fe.fit_transform(X)

print("Original feature count:", X.shape[1])
print("Engineered feature count:", X_fe.shape[1])
print("New columns added:")
print([c for c in X_fe.columns if c not in X.columns])

In [ ]:
y_24 = y['target_24h']
# check if target leaked into features
print([c for c in X_fe.columns if 'target' in c.lower()])

# check correlation with target (quick leak detection)
corr = X_fe.copy()
corr['target'] = y_24

print(
    corr.corr()['target']
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
y_24 = y['target_24h']

# group = event_id
groups = X['event_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, valid_idx = next(gss.split(X_fe, y_24, groups=groups))

X_train = X_fe.iloc[train_idx]
X_valid = X_fe.iloc[valid_idx]
y_train = y_24.iloc[train_idx]
y_valid = y_24.iloc[valid_idx]

model_24 = HistGradientBoostingClassifier(
    max_depth=5,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

model_24.fit(X_train, y_train)

valid_pred = model_24.predict_proba(X_valid)[:, 1]

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_valid, valid_pred)

print("Validation ROC AUC (24h):", round(auc, 4))

In [ ]:
print("Train rows:", len(X_train))
print("Valid rows:", len(X_valid))
print("Unique train events:", X.iloc[train_idx]['event_id'].nunique())
print("Unique valid events:", X.iloc[valid_idx]['event_id'].nunique())

train_events = set(X.iloc[train_idx]['event_id'])
valid_events = set(X.iloc[valid_idx]['event_id'])

print("Overlap in event_id:", len(train_events & valid_events))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# target
y_24 = y['target_24h']
groups = X['event_id']

# split once, reuse same indices for fair comparison
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(gss.split(X, y_24, groups=groups))

# original X for modeling: drop event_id
X_base = X.drop(columns=['event_id']).copy()

# split base features
X_train_base = X_base.iloc[train_idx]
X_valid_base = X_base.iloc[valid_idx]

# split engineered features
X_train_fe = X_fe.iloc[train_idx]
X_valid_fe = X_fe.iloc[valid_idx]

y_train = y_24.iloc[train_idx]
y_valid = y_24.iloc[valid_idx]

# same model for both
model_base = HistGradientBoostingClassifier(
    max_depth=5,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

model_fe = HistGradientBoostingClassifier(
    max_depth=5,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

# train + evaluate base
model_base.fit(X_train_base, y_train)
pred_base = model_base.predict_proba(X_valid_base)[:, 1]
auc_base = roc_auc_score(y_valid, pred_base)

# train + evaluate FE
model_fe.fit(X_train_fe, y_train)
pred_fe = model_fe.predict_proba(X_valid_fe)[:, 1]
auc_fe = roc_auc_score(y_valid, pred_fe)

print("AUC with original features:   ", round(auc_base, 4))
print("AUC with engineered features: ", round(auc_fe, 4))
print("Improvement:                  ", round(auc_fe - auc_base, 4))

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd

result = permutation_importance(
    model_fe,
    X_valid_fe,
    y_valid,
    n_repeats=5,
    random_state=42,
    scoring='roc_auc'
)

feat_imp = pd.DataFrame({
    'feature': X_valid_fe.columns,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(15))

In [ ]:
important_features = feat_imp[feat_imp['importance'] > 0]['feature'].tolist()

X_fe_selected = X_fe[important_features]

In [ ]:
X_train_sel = X_fe_selected.iloc[train_idx]
X_valid_sel = X_fe_selected.iloc[valid_idx]

model_sel = HistGradientBoostingClassifier(
    max_depth=5,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

model_sel.fit(X_train_sel, y_train)

pred_sel = model_sel.predict_proba(X_valid_sel)[:, 1]

from sklearn.metrics import roc_auc_score
auc_sel = roc_auc_score(y_valid, pred_sel)

print("AUC with selected features:", round(auc_sel, 4))
print("Number of features used:", len(important_features))

In [ ]:
X_no_main = X_fe_selected.drop(columns=['dist_min_ci_0_5h'])

In [ ]:
X_input = X_no_main

# split using SAME indices
X_train_tmp = X_input.iloc[train_idx]
X_valid_tmp = X_input.iloc[valid_idx]

# train model
model_tmp = HistGradientBoostingClassifier(
    max_depth=5,
    learning_rate=0.05,
    max_iter=200,
    random_state=42
)

model_tmp.fit(X_train_tmp, y_train)

# evaluate
pred_tmp = model_tmp.predict_proba(X_valid_tmp)[:, 1]

from sklearn.metrics import roc_auc_score
auc_tmp = roc_auc_score(y_valid, pred_tmp)

print("AUC:", round(auc_tmp, 4))
print("Number of features:", X_input.shape[1])

In [ ]:
labels = [
    "Original",
    "After Feature Engineering",
    "Final (Refined Features)"
]

values = [
    auc_base,   # original
    auc_fe,     # after FE
    auc_tmp     # final
]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values)

plt.ylim(0.95, 1.00)
plt.ylabel("Validation ROC AUC")
plt.title("Impact of Feature Engineering and Feature Refinement")

# annotate values
for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.0005,
        f"{value:.4f}",
        ha='center'
    )

plt.tight_layout()
plt.show()

# MODEL

```python
xgb_params = {
    'n_estimators': 200,
    'max_depth': 3,
    'learning_rate': 0.05,
    'random_state': 42
}

base_xgb = xgb.XGBClassifier(**xgb_params)

calibrated_xgb = CalibratedClassifierCV(base_xgb, cv=5)

Model = MultiOutputClassifier(calibrated_xgb)
```

# PIPELINE

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Model)
])

# TRAINING

pipeline.fit(X_input, y)

# SAVE MODEL

```python

model_filename = 'wildfire_xgb_1.pkl'

with open(model_filename, 'wb') as f:
    cloudpickle.dump(pipeline, f)
```

# LOAD MODEL

In [ ]:
file = f"{belbino_wildfire_scikitlearn_default_1_path}/wildfire_xgb_1.pkl"

with open (file,'rb')as f:
    Model =cloudpickle.load(f)

Model

# PREDICTION

In [ ]:
probs_list = Model.predict_proba(test_df)


submission = pd.DataFrame({
    'event_id': test_df['event_id'],
    'prob_12h': probs_list[0][:, 1],
    'prob_24h': probs_list[1][:, 1],
    'prob_48h': probs_list[2][:, 1],
    'prob_72h': probs_list[3][:, 1]
}, index=test_df.index)

prob_cols = ['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']

submission[prob_cols] = np.maximum.accumulate(submission[prob_cols].values, axis=1)

# SUBMISSION

In [ ]:
submission.to_csv('submission.csv', index=False)

In [ ]:
submission